# Lab 6: Time Series Forecasting and Sequence Generation

This repository contains the implementation for Lab 6, focusing on Recurrent Neural Networks (RNNs) for both regression-based time series forecasting and categorical sequence generation.

## Project Structure

* **Task 1: Time Series Forecasting**
    * Implementation of Simple RNNs and Deep RNNs.
    * Comparison of One-Step Ahead vs. Multi-Step Ahead forecasting.
    * Application of Batch Normalization and Layer Normalization.
    * Performance evaluation using LSTM and GRU architectures.
* **Task 2: Bach Chorales Generation**
    * Preprocessing of the Bach Chorales dataset.
    * Training a sequence-to-sequence model to predict musical notes.
    * A generation script to produce "Bach-like" music sequences.

## Requirements
* Python 3.x
* TensorFlow / Keras
* NumPy
* Pandas
* Matplotlib

## How to Run
1. Open the project in PyCharm.
2. Ensure all dependencies are installed via `pip install tensorflow matplotlib numpy`.
3. Run the scripts sequentially to observe the training history and error metrics (MSE/MAE).

### Task 1: Time Series Forecasting
This script covers the creation of a synthetic dataset, training RNNs, LSTMs, and GRUs, and applying normalization.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# --- STEP 0: Data Generation ---
# Generating a synthetic sine wave time series with noise
def generate_time_series(batch_size, n_steps):
    freq1, freq2, offsets1, offsets2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time - offsets1) * (freq1 * 10 + 10))  # wave 1
    series += 0.2 * np.sin((time - offsets2) * (freq2 * 20 + 20)) # wave 2
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)   # noise
    return series[..., np.newaxis].astype(np.float32)

n_steps = 50
series = generate_time_series(10000, n_steps + 1)
X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]
X_test, y_test = series[9000:, :n_steps], series[9000:, -1]

# --- STEP 1 & 2: Deep RNN with Normalization ---
# Building a Deep RNN with Layer Normalization for stability
model_deep_rnn = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.LayerNormalization(), # Step 2: Applying Layer Normalization
    keras.layers.SimpleRNN(20),
    keras.layers.LayerNormalization(),
    keras.layers.Dense(1) # Final output for 1-step prediction
])

model_deep_rnn.compile(loss="mse", optimizer="adam")
history = model_deep_rnn.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

# --- STEP 3: LSTM and GRU ---
# LSTM is better at capturing long-term dependencies than SimpleRNN
model_lstm = keras.models.Sequential([
    keras.layers.LSTM(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.LSTM(20),
    keras.layers.Dense(1)
])

# GRU is a simplified version of LSTM, often faster to train
model_gru = keras.models.Sequential([
    keras.layers.GRU(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.GRU(20),
    keras.layers.Dense(1)
])

# Training and reporting results (MSE)
for model in [model_lstm, model_gru]:
    model.compile(loss="mse", optimizer="adam")
    model.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))
    print(f"Test MSE: {model.evaluate(X_test, y_test)}")